## Import Library

In [ ]:
"""DenseNet-121 multi-label training script for NIH ChestX-ray14.
 
Run on Kaggle with T4 GPU. Outputs:
  - /kaggle/working/multilabel_model.pt
  - /kaggle/working/multilabel_thresholds.json
  - /kaggle/working/training_history.csv
 
References:
  - Strick et al. (2025): Reproducing and Improving CheXNet.
  - Lin et al. (2017): Focal Loss for Dense Object Detection, ICCV 2017.
"""
 
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score
import timm
 
from dataset_utils import (
    ALL_CONDITIONS,
    build_image_index,
    load_multilabel_csv,
    patient_level_split,
    ChestXrayDataset,
    get_train_transform,
    get_eval_transform,
)

## Config

In [ ]:
# Config
BATCH_SIZE      = 64
BATCH_SIZE_EVAL = 32
EPOCHS          = 20
NUM_WORKERS     = 2
LR              = 5e-5
WEIGHT_DECAY    = 1e-2
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH      = Path("/kaggle/working/multilabel_model.pt")
THRESHOLD_PATH  = Path("/kaggle/working/multilabel_thresholds.json")
HISTORY_PATH    = Path("/kaggle/working/training_history.csv")

## Helper Function

In [ ]:
# Loss 
class FocalLoss(nn.Module):
    """Binary focal loss for multi-label imbalanced classification (Lin et al., 2017)."""
 
    def __init__(self, gamma: float = 2.0, pos_weight: torch.Tensor = None):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        pt = torch.exp(-bce)
        return ((1 - pt) ** self.gamma * bce).mean()
 
 
#  Model 
def build_model() -> nn.Module:
    """DenseNet-121 pretrained ImageNet, 14 binary outputs, dropout 0.2."""
    model = timm.create_model(
        "densenet121", pretrained=True, num_classes=14, drop_rate=0.2
    )
    return model.to(DEVICE)
 
 
def compute_pos_weight(df: pd.DataFrame) -> torch.Tensor:
    """Compute per-class pos_weight from training label distribution."""
    counts     = df[ALL_CONDITIONS].sum().values
    neg_counts = len(df) - counts
    weights    = neg_counts / np.maximum(counts, 1)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)
 
 
# Train & Eval 
def train_one_epoch(model, loader, optimizer, criterion, scaler) -> float:
    """Run one training epoch with AMP and return mean loss."""
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)
 
 
def evaluate(model, loader, thresholds=None) -> dict:
    """Return mean and per-class AUC/F1 using pre-allocated arrays."""
    if thresholds is None:
        thresholds = [0.5] * len(ALL_CONDITIONS)
 
    model.eval()
    n          = len(loader.dataset)
    all_probs  = np.empty((n, len(ALL_CONDITIONS)), dtype=np.float32)
    all_labels = np.empty((n, len(ALL_CONDITIONS)), dtype=np.float32)
    idx = 0
 
    with torch.no_grad():
        for images, labels in loader:
            bsz = images.size(0)
            with torch.amp.autocast("cuda"):
                probs = torch.sigmoid(model(images.to(DEVICE))).float().cpu().numpy()
            all_probs[idx:idx+bsz]  = probs
            all_labels[idx:idx+bsz] = labels.numpy()
            idx += bsz
 
    aucs, f1s = [], []
    for i, cond in enumerate(ALL_CONDITIONS):
        if all_labels[:, i].sum() == 0:
            continue
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        f1  = f1_score(
            all_labels[:, i],
            (all_probs[:, i] >= thresholds[i]).astype(int),
            zero_division=0,
        )
        aucs.append(auc)
        f1s.append(f1)
 
    return {
        "mean_auc":      float(np.mean(aucs)),
        "mean_f1":       float(np.mean(f1s)),
        "per_class_auc": dict(zip(ALL_CONDITIONS, aucs)),
        "per_class_f1":  dict(zip(ALL_CONDITIONS, f1s)),
    }
 
 
# Training Loop
def run_training(model, train_loader, val_loader, df_train) -> nn.Module:
    """Full training loop with Focal Loss, AdamW, ReduceLROnPlateau, AUC checkpointing."""
    pos_weight = compute_pos_weight(df_train)
    criterion  = FocalLoss(gamma=2.0, pos_weight=pos_weight)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )
    scaler  = torch.amp.GradScaler("cuda")
    history = []
    best_auc = 0.0
 
    for epoch in range(1, EPOCHS + 1):
        train_loss  = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_metrics = evaluate(model, val_loader)
        scheduler.step(val_metrics["mean_auc"])
        torch.cuda.empty_cache()
 
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f}"
            f" | val_auc={val_metrics['mean_auc']:.4f}"
            f" | val_f1={val_metrics['mean_f1']:.4f}"
            f" | lr={current_lr:.2e}"
        )
 
        history.append({
            "epoch":   epoch,
            "loss":    round(train_loss, 6),
            "val_auc": round(val_metrics["mean_auc"], 6),
            "val_f1":  round(val_metrics["mean_f1"], 6),
            "lr":      current_lr,
        })
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
 
        if val_metrics["mean_auc"] > best_auc:
            best_auc = val_metrics["mean_auc"]
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"Checkpoint saved (val_auc={best_auc:.4f})")
 
    model.load_state_dict(torch.load(MODEL_PATH))
    return model

## Main Run

In [ ]:
# Entry point 
if __name__ == "__main__":
    print(torch.__version__, "|", torch.cuda.get_device_name(0))
 
    IMAGE_INDEX = build_image_index()
    print(f"Image index: {len(IMAGE_INDEX)} files")
 
    df = load_multilabel_csv(IMAGE_INDEX)
    df_train, df_val, df_test = patient_level_split(df)
    print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
 
    train_ds = ChestXrayDataset(df_train, IMAGE_INDEX, get_train_transform())
    val_ds   = ChestXrayDataset(df_val,   IMAGE_INDEX, get_eval_transform())
    test_ds  = ChestXrayDataset(df_test,  IMAGE_INDEX, get_eval_transform())
 
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
 
    model = build_model()
    if MODEL_PATH.exists():
        model.load_state_dict(torch.load(MODEL_PATH))
        print("Loaded existing checkpoint")
    else:
        model = run_training(model, train_loader, val_loader, df_train)